# Repository and AVeriTeC Setup

This notebook prepares the development environment for the evidence-retrieval
project.

It will:

1. connect to the project Git repository;
2. create the minimal repository structure;
3. establish a persistent external data directory;
4. download the AVeriTeC train and development annotations;
5. record the exact Hugging Face revision used;
6. verify that the development data can be loaded.

The full AVeriTeC knowledge store is optional and is not downloaded by default.

## 1. Project Configuration

In [32]:
GITHUB_REPOSITORY = "https://github.com/REALBroomFish/MSc_DissertationProject.git"
GIT_BRANCH = "main"

HF_REPOSITORY = "chenxwh/AVeriTeC"

# The full development knowledge store is approximately 11.5 GB compressed.
DOWNLOAD_FULL_DEV_KNOWLEDGE_STORE = True
EXTRACT_FULL_DEV_KNOWLEDGE_STORE = True

PROJECT_NAME = (
    GITHUB_REPOSITORY.rstrip("/")
    .split("/")[-1]
    .removesuffix(".git")
)

print("Project:", PROJECT_NAME)

Project: MSc_DissertationProject


## 2. Detect if in Google Colab and prepare storage

In [33]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    # Fast, temporary working copy of the Git repository.
    WORK_ROOT = Path("/content")

    # Persistent dataset storage.
    DATA_ROOT = (Path("/content/drive/MyDrive") / "MSc_Dissertation" / "datasets")
else:
    # When running locally, use the current directory or its parent.
    current_directory = Path.cwd().resolve()

    WORK_ROOT = current_directory
    DATA_ROOT = current_directory / ".local_data"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", IN_COLAB)
print("Working directory:", WORK_ROOT)
print("Persistent data directory:", DATA_ROOT)

Running in Colab: False
Working directory: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison
Persistent data directory: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data


## 3. Clone or locate repository

In [34]:
import os
import subprocess
from pathlib import Path


def run_command(command: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and show stdout/stderr if it fails."""
    print("$", " ".join(command))

    result = subprocess.run(command, cwd=cwd, check=False, text=True, capture_output=True)

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}:\n" f"{result.stderr}")

    return result


def find_git_repository(start: Path) -> Path | None:
    """Search the current directory and its parents for a .git directory."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate

    return None


existing_repository = find_git_repository(Path.cwd())

if existing_repository is not None:
    PROJECT_ROOT = existing_repository
    print(f"Using existing repository at:\n{PROJECT_ROOT}")
else:
    PROJECT_ROOT = WORK_ROOT / PROJECT_NAME

    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"{PROJECT_ROOT} already exists and is not empty.")

    run_command(["git", "clone", GITHUB_REPOSITORY, str(PROJECT_ROOT)])

os.chdir(PROJECT_ROOT)

run_command(["git", "status", "--short"], cwd=PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Using existing repository at:
C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison
$ git status --short
?? .gitignore
?? .local_data/
?? DESIGN.md
?? ENVIRONMENT.md
?? README.md
?? data/
?? notebooks/
?? requirements.txt
?? src/

Project root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison


## 4. Create minimal structure 
(will be adjusted as project proceeds)

In [35]:
directories = [PROJECT_ROOT / "notebooks", PROJECT_ROOT / "data", PROJECT_ROOT / "outputs"]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

data_readme = PROJECT_ROOT / "data" / "README.md"

if not data_readme.exists():
    data_readme.write_text("# Data\n\n" "Dataset files are stored outside the Git repository.\n" "The local location is configured through the " "`AVERITEC_ROOT` environment variable.\n", encoding="utf-8")

gitignore_path = PROJECT_ROOT / ".gitignore"

required_gitignore_entries = [".env", ".ipynb_checkpoints/", "__pycache__/", "*.py[cod]", "outputs/", "data/*", "!data/README.md"]

existing_gitignore = (gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else "")

missing_entries = [entry for entry in required_gitignore_entries if entry not in existing_gitignore.splitlines()]

if missing_entries:
    with gitignore_path.open("a", encoding="utf-8") as file:
        if existing_gitignore and not existing_gitignore.endswith("\n"):
            file.write("\n")

        file.write("\n# Local project artefacts\n")
        file.write("\n".join(missing_entries))
        file.write("\n")

print("Minimal repository structure created.")

Minimal repository structure created.


## 5. install dependencies
(will be adjusted to use requirements.txt later)

In [36]:
!py -m pip install -q --upgrade huggingface_hub hf_xet pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 6. Resolve and freeze HuggingFace revision

In [37]:
from huggingface_hub import HfApi, hf_hub_download

hf_api = HfApi()

# AVeriTeC is currently hosted as a regular Hub repository,
# so model_info/default repo type is used rather than repo_type="dataset".
repository_info = hf_api.model_info(HF_REPOSITORY)

HF_REVISION = repository_info.sha

if not HF_REVISION:
    raise RuntimeError("Could not resolve the Hugging Face revision.")

print("Hugging Face repository:", HF_REPOSITORY)
print("Resolved revision:", HF_REVISION)

Hugging Face repository: chenxwh/AVeriTeC
Resolved revision: 2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031


## 7. Download annotation files

In [38]:
DATASET_ROOT = (DATA_ROOT / "averitec" / HF_REVISION[:12])

DATASET_ROOT.mkdir(parents=True, exist_ok=True)

ANNOTATION_FILES = ["data/train.json", "data/dev.json"]

downloaded_files: dict[str, Path] = {}

for filename in ANNOTATION_FILES:
    local_path = Path(hf_hub_download(repo_id=HF_REPOSITORY, filename=filename, revision=HF_REVISION, local_dir=DATASET_ROOT))

    downloaded_files[filename] = local_path
    print(f"Downloaded {filename} -> {local_path}")

Downloaded data/train.json -> C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\data\train.json
Downloaded data/dev.json -> C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\data\dev.json


## 8. Validate annotations

In [39]:
import json

TRAIN_PATH = downloaded_files["data/train.json"]
DEV_PATH = downloaded_files["data/dev.json"]


def load_json(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8") as file:
        records = json.load(file)

    if not isinstance(records, list):
        raise TypeError(f"Expected a list of records in {path}, " f"received {type(records).__name__}.")

    return records


train_records = load_json(TRAIN_PATH)
dev_records = load_json(DEV_PATH)

print(f"Training records: {len(train_records):,}")
print(f"Development records: {len(dev_records):,}")

Training records: 3,068
Development records: 500


### 8.1. Small annotation inspection

In [40]:
first_record = dev_records[0]

expected_fields = {"claim", "label", "justification", "questions"}

missing_fields = expected_fields.difference(first_record)

if missing_fields:
    raise KeyError(
        f"The first development record is missing: {missing_fields}"
    )

print("Available fields:")
print(sorted(first_record.keys()))

print("\nClaim:")
print(first_record["claim"])

print("\nLabel:")
print(first_record["label"])

print("\nNumber of annotated questions:")
print(len(first_record["questions"]))

Available fields:
['cached_original_claim_url', 'claim', 'claim_date', 'claim_types', 'fact_checking_article', 'fact_checking_strategies', 'justification', 'label', 'location_ISO_code', 'original_claim_url', 'questions', 'reporting_source', 'required_reannotation', 'speaker']

Claim:
In a letter to Steve Jobs, Sean Connery refused to appear in an apple commercial.

Label:
Refuted

Number of annotated questions:
2


## 9. Record a dataset manifest

In [41]:
from datetime import datetime, timezone

manifest = {
    "dataset": "AVeriTeC",
    "huggingface_repository": HF_REPOSITORY,
    "revision": HF_REVISION,
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "files": {filename: {"path": str(path.relative_to(DATASET_ROOT)), "size_bytes": path.stat().st_size} for filename, path in downloaded_files.items()},
}

manifest_path = DATASET_ROOT / "dataset_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path.read_text(encoding="utf-8"))

{
  "dataset": "AVeriTeC",
  "huggingface_repository": "chenxwh/AVeriTeC",
  "revision": "2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031",
  "downloaded_at_utc": "2026-08-05T14:14:33.488047+00:00",
  "files": {
    "data/train.json": {
      "path": "data\\train.json",
      "size_bytes": 10184813
    },
    "data/dev.json": {
      "path": "data\\dev.json",
      "size_bytes": 1785475
    }
  }
}


## 10. Register the logcal data location

In [42]:
env_path = PROJECT_ROOT / ".env"

env_path.write_text(f"AVERITEC_ROOT={DATASET_ROOT}\n" f"AVERITEC_REVISION={HF_REVISION}\n", encoding="utf-8")

os.environ["AVERITEC_ROOT"] = str(DATASET_ROOT)
os.environ["AVERITEC_REVISION"] = HF_REVISION

print("Created local configuration:", env_path)
print("AVERITEC_ROOT:", os.environ["AVERITEC_ROOT"])

Created local configuration: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.env
AVERITEC_ROOT: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a


## 11. Download the full development store (optional)

## WARNING: 
the full development archive is ~11.5 GB, and extracting it requires an additional ~35 GB, so only do this if you really want to!! 

In [46]:
import shutil
import zipfile

KNOWLEDGE_STORE_FILENAME = ("data_store/knowledge_store/dev_knowledge_store.zip")

if DOWNLOAD_FULL_DEV_KNOWLEDGE_STORE:
    disk_usage = shutil.disk_usage(DATA_ROOT)

    print("Free persistent storage before download:", f"{disk_usage.free / (1024**3):.2f} GiB")

    knowledge_store_zip = Path(hf_hub_download(repo_id=HF_REPOSITORY, filename=KNOWLEDGE_STORE_FILENAME, revision=HF_REVISION, local_dir=DATASET_ROOT))

    print("Knowledge-store archive:", knowledge_store_zip)
    print("Compressed archive size:", f"{knowledge_store_zip.stat().st_size / (1024**3):.2f} GiB")

    if EXTRACT_FULL_DEV_KNOWLEDGE_STORE:
        extraction_directory = (DATASET_ROOT / "knowledge_store" / "dev")

        with zipfile.ZipFile(knowledge_store_zip) as archive:
            uncompressed_bytes = sum(member.file_size for member in archive.infolist())

            free_bytes = shutil.disk_usage(DATA_ROOT).free

            print("Expected extracted size:", f"{uncompressed_bytes / (1024**3):.2f} GiB")
            print("Currently free:", f"{free_bytes / (1024**3):.2f} GiB")

            # Retain a small safety margin beyond the archive's declared size.
            required_bytes = int(uncompressed_bytes * 1.10)

            if free_bytes < required_bytes:
                raise RuntimeError("Insufficient free storage to extract the " "development knowledge store safely.")

            extraction_directory.mkdir(parents=True,exist_ok=True)

            archive.extractall(extraction_directory)

        print("Extracted to:", extraction_directory)
else:
    print("Full development knowledge store not downloaded. " "Set DOWNLOAD_FULL_DEV_KNOWLEDGE_STORE=True when " "persistent storage has been confirmed.")

Free persistent storage before download: 34.89 GiB
Knowledge-store archive: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\data_store\knowledge_store\dev_knowledge_store.zip
Compressed archive size: 10.75 GiB
Expected extracted size: 34.04 GiB
Currently free: 34.89 GiB


RuntimeError: Insufficient free storage to extract the development knowledge store safely.

## 12. Final Setup check

In [ ]:
print("Repository")
print("----------")
print("Root:", PROJECT_ROOT)

print("\nDataset")
print("-------")
print("Root:", DATASET_ROOT)
print("Revision:", HF_REVISION)
print("Train exists:", TRAIN_PATH.exists())
print("Dev exists:", DEV_PATH.exists())

print("\nSample claim")
print("------------")
print(dev_records[0]["claim"])

print("\nGit status")
print("----------")
run_command(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
)

Repository
----------
Root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison

Dataset
-------
Root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a
Revision: 2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031
Train exists: True
Dev exists: True

Sample claim
------------
In a letter to Steve Jobs, Sean Connery refused to appear in an apple commercial.

Git status
----------
$ git status --short
?? .gitignore
?? .local_data/
?? DESIGN.md
?? ENVIRONMENT.md
?? README.md
?? data/
?? notebooks/
?? requirements.txt
?? src/



CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout='?? .gitignore\n?? .local_data/\n?? DESIGN.md\n?? ENVIRONMENT.md\n?? README.md\n?? data/\n?? notebooks/\n?? requirements.txt\n?? src/\n', stderr='')